<a href="https://colab.research.google.com/github/shuruq18/Covid19-detection-project/blob/main/Covid_19_detection_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===============================
# Step 1: Load & Preprocess Data
# ===============================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load dataset
df = pd.read_csv("/content/ Covid-Dataset.csv")

# -------------------------------
# Basic data cleaning
# -------------------------------

# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df.fillna(df.median(numeric_only=True), inplace=True)

# Encode categorical columns
label_encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# -------------------------------
# Handle outliers using IQR
# -------------------------------
for col in df.select_dtypes(include=["int64", "float64"]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

# -------------------------------
# Split features & target
# -------------------------------
X = df.drop("COVID-19", axis=1)   # change column name if different
y = df["COVID-19"]

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [2]:
print(df.columns.tolist())

['Breathing Problem', 'Fever', 'Dry Cough', 'Sore throat', 'Running Nose', 'Asthma', 'Chronic Lung Disease', 'Headache', 'Heart Disease', 'Diabetes', 'Hyper Tension', 'Fatigue', 'Gastrointestinal', 'Abroad travel', 'Contact with COVID Patient', 'Attended Large Gathering', 'Visited Public Exposed Places', 'Family working in Public Exposed Places', 'Wearing Masks', 'Sanitization from Market', 'COVID-19']


In [3]:
# =======================================
# Step 2: Ensemble Model + Grid Search
# =======================================

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# -------------------------------
# Random Forest
# -------------------------------
rf = RandomForestClassifier(random_state=42)

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf,
    rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_model = rf_grid.best_estimator_

print("Best Parameters:", rf_grid.best_params_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


In [4]:
# ===============================
# Step 3: Model Evaluation
# ===============================

from sklearn.metrics import classification_report, accuracy_score

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        77

    accuracy                           1.00        77
   macro avg       1.00      1.00      1.00        77
weighted avg       1.00      1.00      1.00        77



In [5]:
import joblib

joblib.dump(best_model, "covid_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "encoders.pkl")

print("Model saved successfully!")

Model saved successfully!


In [6]:
pip install pandas numpy scikit-learn joblib streamlit

In [ ]:
!streamlit run "streamlit_covid.py"



2026-07-07 01:38:51.894 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.201.134:8501



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install streamlit -q
!npm install -g localtunnel -q

In [ ]:
!streamlit run "streamlit_covid.py" &>/content/logs.txt &
!npx localtunnel --port 8501

In [ ]:
!wget -q -O - https://loca.lt/mytunnelpassword

In [7]:
import streamlit as st
import pandas as pd
import joblib
import base64

# ===============================
# Page Config
# ===============================
st.set_page_config(
    page_title="COVID-19 Detection",
    layout="centered"
)

# ===============================
# Background Image
# ===============================
def set_background(image_file):
    try:
        with open(image_file, "rb") as f:
            encoded = base64.b64encode(f.read()).decode()

        st.markdown(
            f"""
            <style>
            .stApp {{
                background-image: url("data:image/png;base64,{encoded}");
                background-size: cover;
                background-position: center;
                background-attachment: fixed;
            }}
            </style>
            """,
            unsafe_allow_html=True
        )
    except:
        pass

set_background("background.png")

# ===============================
# Load Files
# ===============================
model = joblib.load("covid_model.pkl")
scaler = joblib.load("scaler.pkl")
encoders = joblib.load("encoders.pkl")

# ===============================
# Title
# ===============================
st.title("🦠 COVID-19 Detection System")
st.markdown("### Machine Learning Based Diagnosis")

# ===============================
# Features
# ===============================
features = [
    "Breathing Problem",
    "Fever",
    "Dry Cough",
    "Sore throat",
    "Running Nose",
    "Asthma",
    "Chronic Lung Disease",
    "Headache",
    "Heart Disease",
    "Diabetes",
    "Hyper Tension",
    "Fatigue ",
    "Gastrointestinal ",
    "Abroad travel",
    "Contact with COVID Patient",
    "Attended Large Gathering",
    "Visited Public Exposed Places",
    "Family working in Public Exposed Places",
    "Wearing Masks",
    "Sanitization from Market"
]

# ===============================
# Input Form
# ===============================
st.subheader("Patient Symptoms")

user_input = {}

for feature in features:
    user_input[feature] = st.selectbox(
        feature.strip(),
        ["No", "Yes"],
        key=feature
    )

# ===============================
# Prediction
# ===============================
if st.button("Predict COVID-19"):

    try:
        input_df = pd.DataFrame([user_input])

        # Encode categorical features
        for col, encoder in encoders.items():

            if col in input_df.columns:

                # تحويل القيم بنفس طريقة التدريب
                input_df[col] = encoder.transform(
                    input_df[col].astype(str)
                )

        # ترتيب الأعمدة بنفس ترتيب التدريب
        input_df = input_df[features]

        # Scale
        input_scaled = scaler.transform(input_df)

        # Predict
        prediction = model.predict(input_scaled)

        if prediction[0] == 1:
            st.error("⚠️ COVID-19 POSITIVE")
        else:
            st.success("✅ COVID-19 NEGATIVE")

    except Exception as e:
        st.error(f"Error: {e}")

2026-07-07 01:38:31.644 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 01:38:31.767 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 01:38:32.519 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-07 01:38:32.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 01:38:32.525 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 01:38:32.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 01:38:32.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn